In [ ]:
# uncomment these and run this cell if needed
!pip install evaluate
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset, Dataset
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import get_scheduler
from transformers import Trainer, TrainingArguments
from datasets import concatenate_datasets
import evaluate
import numpy as np
import requests

In [ ]:
#initalize variables
header1 = "Context: "
header2 = "Question: "
header3 = "Answer: "

column_names1 = "context"
column_names2 = "question"
column_names3 = "statement"

def set_header1(header_value):
  global header1
  header1=header_value

def set_header2(header_value):
  global header2
  header2=header_value

def set_header3(header_value):
  global header3
  header3=header_value

def set_column_names1(column_names):
  global column_names1
  column_names1=column_names

def set_column_names2(column_names):
  global column_names2
  column_names2=column_names

def set_column_names3(column_names):
  global column_names3
  column_names3=column_names

def set_master_columns(col1, col2, col3, header_val1, header_val2, header_val3):
  global column_names1, column_names2, column_names3, header1, header2, header3
  column_names1=col1
  column_names2=col2
  column_names3=col3

  header1=header_val1
  header2=header_val2
  header3=header_val3

In [ ]:
# prepping the dataset
raw_datasets_medical = load_dataset("GM07/medhal-lf") #https://huggingface.co/datasets/GM07/medhal-lf

def label_map(example):
    if example["label"] == "yes" or example["label"] == "PASS" or example["label"] == "true" or example["label"] == "1" or example["label"] == 1:
        example["label"] = 1
    else:
        example["label"] = 0
    return example

raw_datasets_medical["train"] = raw_datasets_medical["train"].select(range(10000))
raw_datasets_medical= raw_datasets_medical.map(label_map)

# convert medical columns to match HaluEval
def convert_medical(example):
    return {
        "context": example["context"],      # citing_prompt -> context
        "question": "What is a true medical fact?",
        "statement": example["statement"],     # holding -> answer
        "label": example["label"]
    }
raw_datasets_medical = raw_datasets_medical.map(convert_medical)




/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007.parquet:   0%|          | 0.00/16.5M [00:00<?, ?B/s]

data/train-00001-of-00007.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/train-00002-of-00007.parquet:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

...

data/train-00005-of-00007.parquet:   0%|          | 0.00/483M [00:00<?, ?B/s]

data/train-00006-of-00007.parquet:   0%|          | 0.00/545M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/771888 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
from datasets import Value
from datasets import DatasetDict
cols = ["context", "question", "statement", "label"]


medical_clean = raw_datasets_medical.remove_columns(
    [c for c in raw_datasets_medical["train"].column_names if c not in cols]
)

medical_clean = medical_clean.cast_column("label", Value("int64"))

#tokenized_datasets = medical_clean.map(tokenize_function, batched=True)
split_datasets = medical_clean["train"].train_test_split(test_size=0.2, seed=42)


Casting the dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = split_datasets["test"].shuffle(seed=42).select(range(100))

In [ ]:
print(small_train_dataset.column_names)
print(small_train_dataset[0])

['context', 'statement', 'label', 'question']
{'context': "A 65-year old male patient with early gastric cancer was transferred from Aruba to our institution. He had a 3-year history of black stools and anemia. His past medical history included multiple comorbidities: diabetes, chronic renal failure, alcoholic cirrhosis Child A, complete heart blockade and thrombocytopenia of unknown etiology. An upper endoscopy and biopsy revealed a well-differentiated intestinal type adenocarcinoma in the antrum. Endoscopic ultrasonography showed a hypoechoic, 3.2 cm neoplasm, without muscularis externa infiltration and reactive ganglia (). Endoscopic mucosal resection was chosen due to tumor size, stage and comorbidities of the patient. The tumor was fully resected without complications. At the end of the procedure the anesthesiologist had difficulty with ventilation and abdominal distention was observed (). He had a 128/91 mmHg blood pressure and 70 bpm heart rate. An endoscopic revision was done b

In [ ]:
rag_docs = load_dataset("qiaojin/PubMedQA", "pqa_artificial") #https://huggingface.co/datasets/qiaojin/PubMedQA/viewer/pqa_artificial?row=0

README.md: 0.00B [00:00, ?B/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/233M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211269 [00:00<?, ? examples/s]

In [ ]:
#knowledge_base = small_train_dataset["context"]
"""
knowledge_base = rag_docs["train"]["context"]
def add_rag_context(example):
    query = str(example["context"])
    query_words = set(query.lower().split())

    best_context = ""
    best_score = 0

    for context in knowledge_base:
        context_text = str(context)

        context_words = set(context_text.lower().split())
        score = len(query_words.intersection(context_words))

        if score > best_score:
            best_score = score
            best_context = context_text

    example["rag_context"] = best_context
    return example
  """

'\nknowledge_base = rag_docs["train"]["context"]\ndef add_rag_context(example):\n    query = str(example["context"])\n    query_words = set(query.lower().split())\n\n    best_context = ""\n    best_score = 0\n\n    for context in knowledge_base:\n        context_text = str(context)\n\n        context_words = set(context_text.lower().split())\n        score = len(query_words.intersection(context_words))\n\n        if score > best_score:\n            best_score = score\n            best_context = context_text\n\n    example["rag_context"] = best_context\n    return example\n  '

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util
import torch

In [ ]:
def chunk_text(text, max_tokens=100):
    token_ids = tokenizer.encode(str(text), add_special_tokens=False)

    return [token_ids[i:i + max_tokens] for i in range(0, len(token_ids), max_tokens)]

In [ ]:
knowledge_base_tokens  = []

for context in rag_docs["train"]["context"]:
    knowledge_base_tokens.extend(chunk_text(context, max_tokens=100))

In [ ]:
knowledge_base = [
    tokenizer.decode(chunk, skip_special_tokens=True)
    for chunk in knowledge_base_tokens
]

In [ ]:
len(knowledge_base)

1199223

In [ ]:

embedder = SentenceTransformer("NeuML/pubmedbert-base-embeddings")


kb_embeddings = embedder.encode(
    knowledge_base,
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=64
)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/18738 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#Save embeddings
torch.save(kb_embeddings.cpu(), "/content/drive/MyDrive/kb_embeddings.pt")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

kb_embeddings = torch.load("/content/drive/MyDrive/kb_embeddings.pt")
kb_embeddings = kb_embeddings.to(device)

In [ ]:

def add_rag_context(example):
    query = str(example["context"])

    if query == "":
        query = str(example["statement"])

    query_embedding = embedder.encode(
        query,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(query_embedding, kb_embeddings)

    #take top 3 from the single row
    top_indices = torch.topk(scores[0], k=3).indices

    combined = "\n".join(knowledge_base[idx] for idx in top_indices)

    #get token count
    token_ids = tokenizer.encode(combined, add_special_tokens=False)
    token_count = len(token_ids)

    example["rag_context"] = combined
    return example

In [ ]:
small_train_dataset_rag = small_train_dataset.map(add_rag_context)
small_eval_dataset_rag = small_eval_dataset.map(add_rag_context)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [ ]:
def tokenize_function(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            header1 + (q if q is not None else ""),
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for q, p, a in zip(examples[column_names1], examples[column_names2], examples[column_names3])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )

In [ ]:
def tokenize_function_no_context(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for p, a in zip(examples[column_names2], examples[column_names3])
    ]

    return tokenizer(
        [c[0] for c in combined],
        padding="max_length",
        truncation=True
    )

In [ ]:
def tokenize_rag_function(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            header1 + (q if q is not None else "") + "" + (r if a is not None else ""),
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for q, p, a, r in zip(examples[column_names1], examples[column_names2], examples[column_names3], examples["rag_context"])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )


In [ ]:
def tokenize_rag_function_no_orginal_context(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            (r if a is not None else ""),
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for q, p, a, r in zip(examples[column_names1], examples[column_names2], examples[column_names3], examples["rag_context"])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )


In [ ]:
tokenized_train_dataset_rag = small_train_dataset_rag.map(tokenize_rag_function, batched=True)
tokenized_eval_dataset_rag = small_eval_dataset_rag.map(tokenize_rag_function, batched=True)

tokenized_train_dataset_rag_no_orginal_context = small_train_dataset_rag.map(tokenize_rag_function_no_orginal_context, batched=True)
tokenized_eval_dataset_rag_no_orginal_context = small_eval_dataset_rag.map(tokenize_rag_function_no_orginal_context, batched=True)


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [ ]:


#no rag
tokenized_train_dataset_not_rag = small_train_dataset_rag.map(tokenize_function, batched=True)
tokenized_eval_dataset_not_rag = small_eval_dataset_rag.map(tokenize_function, batched=True)

tokenized_train_dataset_not_rag_no_context = small_train_dataset_rag.map(tokenize_function_no_context, batched=True)
tokenized_eval_dataset_not_rag_context = small_eval_dataset_rag.map(tokenize_function_no_context, batched=True)


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

#added average binary because it tells the model to Treat this as a binary classification problem and focus on the positive class (1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"],
    }


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate =2e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset_rag,
    eval_dataset=tokenized_eval_dataset_rag,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.407625,0.820000,0.734375,0.979167,0.839286
2,No log,0.346015,0.880000,0.860000,0.895833,0.877551
3,No log,0.366918,0.870000,0.857143,0.875000,0.865979
4,No log,0.299649,0.910000,0.897959,0.916667,0.907216
5,No log,0.369457,0.910000,0.914894,0.895833,0.905263
6,No log,0.342941,0.910000,0.897959,0.916667,0.907216
7,No log,0.358615,0.900000,0.880000,0.916667,0.897959
8,No log,0.374715,0.900000,0.895833,0.895833,0.895833


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.3747148811817169,
 'eval_accuracy': 0.9,
 'eval_precision': 0.8958333333333334,
 'eval_recall': 0.8958333333333334,
 'eval_f1': 0.8958333333333334,
 'eval_runtime': 0.7788,
 'eval_samples_per_second': 128.408,
 'eval_steps_per_second': 16.693,
 'epoch': 8.0}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate =2e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset_rag_no_orginal_context,
    eval_dataset=tokenized_eval_dataset_rag_no_orginal_context,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.653317,0.600000,0.558824,0.791667,0.655172
2,No log,0.653824,0.610000,0.561644,0.854167,0.677686
3,No log,0.845200,0.590000,0.571429,0.583333,0.577320
4,No log,0.832477,0.590000,0.561404,0.666667,0.609524
5,No log,1.053091,0.580000,0.560000,0.583333,0.571429
6,No log,1.052209,0.620000,0.586207,0.708333,0.641509
7,No log,1.200713,0.620000,0.604167,0.604167,0.604167
8,No log,1.213152,0.630000,0.612245,0.625000,0.618557


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 1.213152289390564,
 'eval_accuracy': 0.63,
 'eval_precision': 0.6122448979591837,
 'eval_recall': 0.625,
 'eval_f1': 0.6185567010309279,
 'eval_runtime': 0.7841,
 'eval_samples_per_second': 127.537,
 'eval_steps_per_second': 16.58,
 'epoch': 8.0}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate =2e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset_not_rag,
    eval_dataset=tokenized_eval_dataset_not_rag,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.400910,0.830000,0.746032,0.979167,0.846847
2,No log,0.271620,0.910000,0.854545,0.979167,0.912621
3,No log,0.295007,0.890000,0.893617,0.875000,0.884211
4,No log,0.387008,0.900000,0.895833,0.895833,0.895833
5,No log,0.479107,0.900000,0.931818,0.854167,0.891304
6,No log,0.349556,0.920000,0.884615,0.958333,0.920000
7,No log,0.343175,0.930000,0.918367,0.937500,0.927835
8,No log,0.356944,0.930000,0.918367,0.937500,0.927835


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.3569437861442566,
 'eval_accuracy': 0.93,
 'eval_precision': 0.9183673469387755,
 'eval_recall': 0.9375,
 'eval_f1': 0.9278350515463918,
 'eval_runtime': 0.7818,
 'eval_samples_per_second': 127.904,
 'eval_steps_per_second': 16.627,
 'epoch': 8.0}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate =2e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset_not_rag_no_context,
    eval_dataset=tokenized_eval_dataset_not_rag_context,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.692400,0.520000,0.000000,0.000000,0.000000
2,No log,0.693270,0.480000,0.480000,1.000000,0.648649
3,No log,0.699152,0.480000,0.480000,1.000000,0.648649
4,No log,0.693324,0.480000,0.480000,1.000000,0.648649
5,No log,0.694056,0.480000,0.480000,1.000000,0.648649
6,No log,0.694620,0.480000,0.480000,1.000000,0.648649
7,No log,0.693982,0.480000,0.480000,1.000000,0.648649
8,No log,0.693396,0.480000,0.480000,1.000000,0.648649


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.6933956742286682,
 'eval_accuracy': 0.48,
 'eval_precision': 0.48,
 'eval_recall': 1.0,
 'eval_f1': 0.6486486486486487,
 'eval_runtime': 0.788,
 'eval_samples_per_second': 126.897,
 'eval_steps_per_second': 16.497,
 'epoch': 8.0}